In [1]:
import sys
sys.path.append("..")  # permite importar de src/, que fica uma pasta acima

import pandas as pd
from src.metricas import metricas

dados = pd.read_csv("../data/btc_diario.csv")  # lê o CSV diário
dados.head()

,data,open,high,low,close,volume
0,2018-01-01,10053.719399,10083.254753,9952.777161,10035.471708,367.585433
1,2018-01-02,9913.587741,9991.399105,9845.087716,9936.473297,173.613174
2,2018-01-03,10038.828403,10039.339963,9977.721840,10016.518417,233.209851
3,2018-01-04,10098.323926,10153.629022,10069.642495,10115.574889,345.516869
4,2018-01-05,9930.623518,9964.407676,9853.028099,9925.471370,206.391351


In [2]:
dados["alvo"] = dados["close"] . shift (-1) #define como alvo o fechamento do dia seguinte (o shift "puxa" o valor uma linha pra cima)
dados.head()

,data,open,high,low,close,volume,alvo
0,2018-01-01,10053.719399,10083.254753,9952.777161,10035.471708,367.585433,9936.473297
1,2018-01-02,9913.587741,9991.399105,9845.087716,9936.473297,173.613174,10016.518417
2,2018-01-03,10038.828403,10039.339963,9977.721840,10016.518417,233.209851,10115.574889
3,2018-01-04,10098.323926,10153.629022,10069.642495,10115.574889,345.516869,9925.471370
4,2018-01-05,9930.623518,9964.407676,9853.028099,9925.471370,206.391351,9800.253419


In [3]:
dados.tail()

,data,open,high,low,close,volume,alvo
3233,2026-11-08,19688.561819,19744.179952,19547.892335,19678.755089,669.570116,19635.282013
3234,2026-11-09,19641.711478,19702.350734,19587.736180,19635.282013,118.424774,19694.861389
3235,2026-11-10,19655.326341,19876.245974,19549.795770,19694.861389,263.041596,19809.594139
3236,2026-11-11,19740.696877,19832.886653,19754.850802,19809.594139,723.972269,19888.722316
3237,2026-11-12,19854.014144,19966.522318,19830.942825,19888.722316,795.654277,NaN


In [4]:
dados = dados.dropna() #limpa as linhas que possuem alguma coluna vazia 
dados.tail()


,data,open,high,low,close,volume,alvo
3232,2026-11-07,19614.637252,19616.003570,19375.062634,19586.778970,553.630630,19678.755089
3233,2026-11-08,19688.561819,19744.179952,19547.892335,19678.755089,669.570116,19635.282013
3234,2026-11-09,19641.711478,19702.350734,19587.736180,19635.282013,118.424774,19694.861389
3235,2026-11-10,19655.326341,19876.245974,19549.795770,19694.861389,263.041596,19809.594139
3236,2026-11-11,19740.696877,19832.886653,19754.850802,19809.594139,723.972269,19888.722316


In [5]:
dados["media_7d"] = dados["close"].rolling(7).mean()
dados.head(8)

,data,open,high,low,close,volume,alvo,media_7d
0,2018-01-01,10053.719399,10083.254753,9952.777161,10035.471708,367.585433,9936.473297,NaN
1,2018-01-02,9913.587741,9991.399105,9845.087716,9936.473297,173.613174,10016.518417,NaN
2,2018-01-03,10038.828403,10039.339963,9977.721840,10016.518417,233.209851,10115.574889,NaN
3,2018-01-04,10098.323926,10153.629022,10069.642495,10115.574889,345.516869,9925.471370,NaN
4,2018-01-05,9930.623518,9964.407676,9853.028099,9925.471370,206.391351,9800.253419,NaN
5,2018-01-06,9805.315175,9816.829391,9748.703297,9800.253419,515.472966,9818.037459,NaN
6,2018-01-07,9827.563362,9850.362670,9702.966381,9818.037459,636.602419,9791.413200,9949.685794
7,2018-01-08,9798.011764,9924.767050,9782.689389,9791.413200,439.095808,9794.733084,9914.820293


In [6]:
dados["data"] = pd.to_datetime(dados["data"])

FIM_TREINO = "2024-12-31"  # Data de corte provisória: treino até o fim de 2024, teste depois

treino = dados[dados["data"] <= FIM_TREINO ] 
teste = dados[dados["data"] > FIM_TREINO ]

print(len(treino), len(teste))


2557 680


In [7]:
# Baseline "amanhã = hoje"

y_real = teste["alvo"] # preço de amanhã (o que queremos acertar)
y_prev = teste["close"] # preço de hoje (a previsão do baseline)

resultado_naive = metricas (y_real, y_prev)
resultado_naive

{'MAE': 80.42923186342836,
 'RMSE': 99.68150424153964,
 'MAPE_%': 0.49818302538571946}

In [8]:
# Baseline "média móvel de 7 dias"
y_real = teste["alvo"]
y_prev = teste["media_7d"]

resultado_media7 = metricas(y_real, y_prev)
resultado_media7

{'MAE': 138.26862378245227,
 'RMSE': 169.00855892128928,
 'MAPE_%': 0.8536459822667032}

## Conclusão dos baselines

Comparação no período de teste (a partir de 01/01/2025; data de corte provisória):

1) Naive (amanhã = hoje) | MAE = 80,43 | RMSE = 99,68 | MAPE_% = 0,50 

2) Média móvel de 7 dias | MAE = 138,27 | RMSE = 169,01 | MAPE_% = 0,85 

O baseline naive errou menos que a média móvel em todas as métricas. A média de 7 dias
usa preços mais antigos, então fica "atrasada" em relação ao preço de hoje, que é a
informação mais recente. Qualquer modelo treinado nas próximas etapas só se justifica se
conseguir um MAE menor que 80,43 neste mesmo período de teste.